<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/00_setup_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00: Verificación del Entorno — Landslide4Sense

Verifica que el entorno, dataset y módulos estén correctamente configurados antes de continuar.

![Vista previa: patch multispectral 128×128 con 14 canales (Sentinel-2, SAR, DEM, RedEdge)](https://raw.githubusercontent.com/apmontesp/Landslides_-Applied-ML-Course/main/docs/figures/nb00_patch_preview.png)

*Vista previa: patch multispectral 128×128 con 14 canales (Sentinel-2, SAR, DEM, RedEdge)*

In [ ]:
from google.colab import drive
import os, sys, h5py, subprocess
import numpy as np
from pathlib import Path

# Instalación de dependencias
packages = ['timm', 'segmentation-models-pytorch', 'h5py', 'tqdm']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Montar Drive
drive.mount('/content/drive')

# Rutas de datos (misma estructura que notebooks 02-06)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    DATA_ROOT  = train_img_dir.parent.parent
    img_list   = sorted(list(train_img_dir.glob('*.h5')))
    mask_list  = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'Dataset : {DATA_ROOT}')
    print(f'Muestras: {len(img_list)}')
else:
    print('ERROR: No se encontro TrainData en Drive.')
    print('Verifica que la carpeta Landslide4Sense este en MyDrive con la estructura:')
    print('  MyDrive/Landslide4Sense/TrainData/img/*.h5')
    print('  MyDrive/Landslide4Sense/TrainData/mask/*.h5')


## 1. Verificar estructura del dataset

In [ ]:
partitions = {
    'TrainData': {'img': True,  'mask': True},
    'ValidData': {'img': True,  'mask': False},
    'TestData' : {'img': True,  'mask': False},
}

all_ok = True
for part, dirs in partitions.items():
    for subdir, required in dirs.items():
        p = DATA_ROOT / part / subdir
        n = len(list(p.glob('*.h5'))) if p.exists() else -1
        status = '✓' if n > 0 else ('✗' if required else '—')
        print(f'  {status}  {part}/{subdir:<10} → {n if n >= 0 else "no existe":>5} archivos .h5')
        if required and n <= 0:
            all_ok = False

print()
if all_ok:
    print('✓ Dataset correctamente estructurado — listo para entrenar')
else:
    print('⚠️  Dataset incompleto. Verifica la estructura en Drive.')


## 2. Verificación de imports

In [ ]:
import torch
import timm
import segmentation_models_pytorch as smp
import sklearn

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'timm   : {timm.__version__}')
print(f'smp    : {smp.__version__}')
print(f'sklearn: {sklearn.__version__}')
print(f'h5py   : {h5py.__version__}')
print(f'numpy  : {np.__version__}')
print('\n✓ Todos los imports correctos')


## 3. Visualización de un patch de muestra

In [ ]:
import matplotlib.pyplot as plt

print(f'Total patches TrainData: {len(img_list)}')

with h5py.File(img_list[0], 'r') as f:
    patch = f['img'][:]
with h5py.File(mask_list[0], 'r') as f:
    mask = f['mask'][:]

print(f'Patch shape : {patch.shape}  (H x W x C)')
print(f'Mask shape  : {mask.shape}')
print(f'Dtype       : {patch.dtype}')
print(f'Pixeles deslizamiento: {int(mask.sum())} / {mask.size} ({mask.mean():.2%})')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
channels = [(0, 'Ch0 — S2 Azul'), (7, 'Ch7 — SAR VV'), (9, 'Ch9 — DEM'), (13, 'Ch13 — RedEdge3')]
for ax, (ch, name) in zip(axes, channels):
    ax.imshow(patch[:, :, ch], cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.suptitle(f'Patch: {img_list[0].name}  |  Mask positiva: {mask.sum():.0f} px', fontsize=11)
plt.tight_layout()
plt.show()


## 4. Balance de clases (muestra rápida)

In [ ]:
from tqdm.auto import tqdm

N = min(200, len(mask_list))
positivos = 0
for mp in tqdm(mask_list[:N], desc='Verificando etiquetas'):
    with h5py.File(mp, 'r') as f:
        positivos += int(f['mask'][:].max() > 0)

print(f'\nMuestra: {N} patches')
print(f'  Con deslizamiento   : {positivos} ({positivos/N:.1%})')
print(f'  Sin deslizamiento   : {N-positivos} ({(N-positivos)/N:.1%})')
print(f'\npos_weight configurado: 0.703 (n_neg/n_pos del dataset completo)')


## Resumen

Si todas las celdas se ejecutaron sin errores, el entorno está listo. Continúa en orden:

| Notebook | Descripción |
|----------|-------------|
| `01_eda_analysis.ipynb` | Análisis exploratorio de datos |
| `02_preprocessing.ipynb` | Preprocesamiento y augmentación |
| `03_baseline_rf.ipynb` | Baseline Random Forest |
| `04_resnet50.ipynb` | ResNet-50 fine-tuning |
| `05_efficientnet_b4.ipynb` | EfficientNet-B4 fine-tuning |
| `06_unet_segmentation.ipynb` | U-Net segmentación pixel-level |
| `07_evaluation_comparison.ipynb` | Comparación final de modelos |